In [1]:
import os
import torch
from chebai.result.prediction import Predictor

os.chdir(r"D:\python-chebai")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

c:\Users\mariu\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cpu


In [2]:
# specify the checkpoint path
checkpoint_path = os.path.join("D:\\ba\\neu\\99\\run-20261111_021924-pst7wzsa\\files\\checkpoints\\best_micro_f1_epoch=151_val_loss=0.0110_val_macro-f1=0.6575_val_micro-f1=0.8969.ckpt")

# initialize the predictor - this class loads the data module parameters and the checkpoint
predictor = Predictor(checkpoint_path=checkpoint_path, compile_model=False)
# you can override parameters if needed (e.g., if the splits file has a different location)
#predictor._dm.splits_file_path = os.path.join("path", "to", "splits_file.csv")
print(type(predictor._dm).__name__)
print(predictor._dm.reader.name())
print(predictor._dm.processed_dir)
print(predictor._dm.processed_file_names)
print(predictor._dm_hparams)

--------------------------------------------------
Using device: cpu
For Loaded checkpoint from: D:\ba\neu\99\run-20261111_021924-pst7wzsa\files\checkpoints\best_micro_f1_epoch=151_val_loss=0.0110_val_macro-f1=0.6575_val_micro-f1=0.8969.ckpt
Below are the modules loaded from the checkpoint:
********** Loaded datamodule class: ChEBIOver50SPE
Input dimension for the model: 615 Output dimension for the model: 1500
********** Loaded model class: Electra
Loaded 1500 classification labels.
--------------------------------------------------
ChEBIOver50SPE
smiles_spe
data\chebi_v244\ChEBI50\processed\run_20261111_123404_d897eb4291154279a458b1016d60f938\smiles_spe
['data.pt']
{'_class_path': 'chebai.preprocessing.datasets.chebi.ChEBIOver50SPE', 'chebi_version_train': None, 'single_class': None, 'subset': None, 'apply_label_filter': None, 'apply_id_filter': None, 'batch_size': 32, 'test_split': 0.1, 'validation_split': 0.05, 'prediction_kind': 'test', 'data_limit': None, 'label_filter': None, 'b

In [3]:
# Predict SMILES strings from file or directly
# predictor.predict_from_file(os.path.join("path", "to", "smiles_file.txt"))
preds_smiles = predictor.predict_smiles(
    [
        "OC[C@H](COP(=O)([O-])OCC[N+](C)(C)C)O",
        "CCCCC[C@H](O)CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1OP(=O)(O)O",
    ]
)
for i, pred in enumerate(preds_smiles):
    print(f"SMILES {i}")
    for p, label in zip(pred, predictor._classification_labels):
        if p > 0.5:
            print(f"  Predicted CHEBI:{label} with {p:.4f}")

[transformers] `sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


SMILES 0
  Predicted CHEBI:16247 with 0.5246
  Predicted CHEBI:18059 with 0.5717
  Predicted CHEBI:23213 with 0.9935
  Predicted CHEBI:23217 with 0.9933
  Predicted CHEBI:24870 with 0.9978
  Predicted CHEBI:25697 with 0.9912
  Predicted CHEBI:25699 with 0.9980
  Predicted CHEBI:25703 with 0.9998
  Predicted CHEBI:25710 with 0.9996
  Predicted CHEBI:25806 with 0.9998
  Predicted CHEBI:26020 with 0.9998
  Predicted CHEBI:26079 with 1.0000
  Predicted CHEBI:26082 with 0.9999
  Predicted CHEBI:26816 with 0.8936
  Predicted CHEBI:33241 with 1.0000
  Predicted CHEBI:33285 with 0.9996
  Predicted CHEBI:33302 with 1.0000
  Predicted CHEBI:33304 with 0.9999
  Predicted CHEBI:33579 with 1.0000
  Predicted CHEBI:33582 with 1.0000
  Predicted CHEBI:33675 with 1.0000
  Predicted CHEBI:33702 with 0.9922
  Predicted CHEBI:35267 with 0.9932
  Predicted CHEBI:35274 with 0.9980
  Predicted CHEBI:35701 with 0.9997
  Predicted CHEBI:35741 with 0.6119
  Predicted CHEBI:36313 with 0.6906
  Predicted CHEBI:3

In [4]:
# Use the exact processed training run and its saved test split.
# For older checkpoints or copied data, set this to the original run directory.
processed_run_dir = r"D:\ba\neu\processed\run_20261111_021403_48716b98e8e443889ca11bac75d0cc80"  # e.g. r"D:\ba\data\chebi_v244\ChEBI50\processed\run_..."
splits_file_path = None  # set this if training used an external splits CSV
preds, labels = predictor.evaluate_predictions(
    partition="test",  # or "validation"
    processed_run_dir=processed_run_dir,
    splits_file_path=splits_file_path,
)
print(preds.shape)
print(labels.shape)

Loading test data... (datamodule.current_epoch=N/A)



Loading splits from D:\ba\neu\processed\run_20261111_021403_48716b98e8e443889ca11bac75d0cc80\splits.csv...
torch.Size([19158, 1500])
torch.Size([19158, 1500])


In [5]:
# Predict the whole test set (and also get the labels for evaluation)
# to get the validation set predictions, just replace test_dataloader with val_dataloader
test_dl = predictor._dm.test_dataloader()
preds = []
labels = []
for batch_idx, batch in enumerate(test_dl):
    batch_preds = predictor._model.test_step(batch, batch_idx)
    preds.append(batch_preds["preds"])
    labels.append(batch_preds["labels"])
preds = torch.cat(preds)
labels = torch.cat(labels)
print(preds.shape)
print(labels.shape)

Loading test data... (datamodule.current_epoch=N/A)


: 

In [10]:
# print the metrics
from torchmetrics import F1Score, Recall, Precision

micro_f1 = F1Score(num_labels=preds.shape[1], task="multilabel", average="micro")
micro_precision = Precision(
    num_labels=preds.shape[1], task="multilabel", average="micro"
)
micro_recall = Recall(num_labels=preds.shape[1], task="multilabel", average="micro")
print(f"Micro F1: {micro_f1(preds, labels)}")
print(f"Micro Precision: {micro_precision(preds, labels)}")
print(f"Micro Recall: {micro_recall(preds, labels)}")
macro_f1 = F1Score(num_labels=preds.shape[1], task="multilabel", average="macro")
macro_precision = Precision(
    num_labels=preds.shape[1], task="multilabel", average="macro"
)
macro_recall = Recall(num_labels=preds.shape[1], task="multilabel", average="macro")
print(f"Macro F1: {macro_f1(preds, labels)}")
print(f"Macro Precision: {macro_precision(preds, labels)}")
print(f"Macro Recall: {macro_recall(preds, labels)}")

Micro F1: 0.8946452140808105
Micro Precision: 0.9048358201980591
Micro Recall: 0.8846815824508667
Macro F1: 0.6710875630378723
Macro Precision: 0.7138133645057678
Macro Recall: 0.6718086004257202
